# PRISM — Ragas Five-Metric Evaluation (Google Colab)

This notebook evaluates existing Q&A responses using the original research paper (PDF). RAG and non-RAG responses are compared using **the same questions and evaluation evidence**.

**Important:** Faithfulness measures support from the selected PDF evidence. Context Precision and Context Recall assess the **evaluation-specific TF-IDF retrieval**, not the PRISM system's actual retriever. Answer Correctness depends on the reference answers.


In [ ]:
!pip -q install "ragas==0.2.15" "langchain-openai>=0.2,<0.4" "pymupdf>=1.24" "scikit-learn>=1.3" "pandas>=2.0"

## 1. Select input files by number (upload them to the Colab Files panel first)


In [ ]:
from pathlib import Path
import pandas as pd, numpy as np
BASE=Path('/content')
def choose(ext,label,exclude=None):
    paths=sorted(p for p in BASE.glob('*'+ext) if p.is_file() and p!=exclude)
    if not paths: raise FileNotFoundError(f'{label}: {ext} files found. Upload the required file(s) using the Colab Files panel.')
    print(f'\n{label}:')
    for i,p in enumerate(paths,1): print(f'[{i}] {p.name}')
    while True:
        try:
            k=int(input(f'{label} number: '))
            if 1<=k<=len(paths): return paths[k-1]
        except ValueError: pass
        print('Enter a valid number from the list.')
PDF=choose('.pdf','Source research paper (PDF)')
RAG_CSV=choose('.csv','RAG answers (CSV)')
NONRAG_CSV=choose('.csv','Non-RAG answers (CSV)',exclude=RAG_CSV)
print('\nUse an optional reference-answer CSV? yes/no')
REF_CSV=choose('.csv','Reference answers (CSV)') if input('> ').strip().lower() in ('yes','y') else None
print('Selected files:',PDF.name,RAG_CSV.name,NONRAG_CSV.name,'Reference:',REF_CSV.name if REF_CSV else 'None')

## 2. Validate question alignment and configure reference answers

The optional reference CSV must contain `No`, `Question`, and `Answer` columns. If cloud-generated answers are used as references, treat them as **provisional references until manually verified**.


In [ ]:
def loadqa(path, name):
    d=pd.read_csv(path,encoding='utf-8-sig')
    required={'No','Question','Answer'}
    if not required.issubset(d.columns): raise ValueError(f'{name}: Required columns {required}  are missing: {d.columns.tolist()}')
    d=d[['No','Question','Answer']].copy()
    if d['No'].isna().any() or d['No'].duplicated().any(): raise ValueError(f'{name}: Missing or duplicate No values')
    d['No']=d['No'].astype(int)
    return d.rename(columns={'Question':f'Question_{name}','Answer':f'Answer_{name}'})
rag=loadqa(RAG_CSV,'RAG'); non=loadqa(NONRAG_CSV,'NonRAG')
df=rag.merge(non,on='No',validate='one_to_one')
if len(df)!=len(rag) or len(df)!=len(non): raise ValueError('Question numbers do not match between RAG and non-RAG files')
if not (df.Question_RAG.fillna('').str.strip()==df.Question_NonRAG.fillna('').str.strip()).all():
    raise ValueError('Question text differs between RAG and non-RAG files')
df=df.rename(columns={'Question_RAG':'Question'}).drop(columns='Question_NonRAG')
if REF_CSV:
    ref=loadqa(REF_CSV,'REF')
    df=df.merge(ref,on='No',validate='one_to_one')
    if df.Answer_REF.isna().any(): raise ValueError('Reference answers contain missing values')
    if not (df.Question.str.strip()==df.Question_REF.fillna('').str.strip()).all(): raise ValueError('Question text differs between reference and response CSV files')
    df=df.drop(columns='Question_REF')
print('Number of questions:',len(df),'Reference answers provided:',bool(REF_CSV))
display(df.head(3))

## 3. Extract PDF text and retrieve candidate evidence using the questions only


In [ ]:
import fitz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
TOP_K=5
passages=[]
with fitz.open(PDF) as doc:
    for pn,page in enumerate(doc,1):
        # Exclude the final page (assumed to contain only references for this source paper).
        if pn==len(doc): continue
        blocks=[x for x in page.get_text('blocks') if len(x)>4 and isinstance(x[4],str) and x[4].strip()]
        blocks.sort(key=lambda x:(0 if (x[0]+x[2])/2<page.rect.width/2 else 1,x[1]))
        for block in blocks:
            words=' '.join(block[4].split()).split()
            for start in range(0,len(words),100):
                segment=' '.join(words[start:start+140])
                if len(segment)>=80: passages.append({'page':pn,'text':segment})
if not passages: raise ValueError('Could not extract text from the PDF')
vectorizer=TfidfVectorizer(ngram_range=(1,2),stop_words='english',sublinear_tf=True)
mat=vectorizer.fit_transform([p['text'] for p in passages])
sims=cosine_similarity(vectorizer.transform(df.Question.fillna('')),mat)
contexts={}; audit=[]
for i,row in df.reset_index(drop=True).iterrows():
    picks=sims[i].argsort()[::-1][:TOP_K]
    contexts[int(row.No)]=[f'[PDF p.{passages[j]["page"]}] {passages[j]["text"]}' for j in picks]
    for rank,j in enumerate(picks,1):
        audit.append({'No':int(row.No),'Question':row.Question,'rank':rank,'page':passages[j]['page'],
                      'retrieval_score':float(sims[i,j]),'evidence':passages[j]['text']})
audit_df=pd.DataFrame(audit)
audit_df.to_csv('/content/prism_evidence_audit.csv',index=False,encoding='utf-8-sig')
print('Candidate evidence passages:',len(audit_df),' total. Verify that the evidence is appropriate for each question.')
display(audit_df.head(10))

## 4. Evaluate with Ragas

- Without reference answers: **Answer Relevancy and Faithfulness** (2 metrics).
- With reference answers: **Answer Correctness, Answer Relevancy, Faithfulness, Context Precision, and Context Recall** (5 metrics).

**Context metrics based on evaluation-specific PDF evidence do not measure the PRISM retriever.** This step uses the OpenAI API and may incur charges.


In [ ]:
import os,getpass
if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY']=getpass.getpass('OpenAI API key (hidden): ')
from ragas import EvaluationDataset, evaluate
from ragas.metrics import Faithfulness, AnswerRelevancy, AnswerCorrectness, LLMContextPrecisionWithReference, LLMContextRecall
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
llm=LangchainLLMWrapper(ChatOpenAI(model='gpt-4o-mini',temperature=0))
embeddings=LangchainEmbeddingsWrapper(OpenAIEmbeddings(model='text-embedding-3-small'))
metrics=[AnswerRelevancy(),Faithfulness()]
if REF_CSV:
    metrics += [AnswerCorrectness(),LLMContextPrecisionWithReference(),LLMContextRecall()]
print('Evaluation metrics:',[m.name for m in metrics])

In [ ]:
samples=[]; labels=[]
for _,row in df.iterrows():
    for cond,field in [('RAG','Answer_RAG'),('Non-RAG','Answer_NonRAG')]:
        item={'user_input':str(row.Question),'response':str(row[field]),'retrieved_contexts':contexts[int(row.No)]}
        if REF_CSV: item['reference']=str(row.Answer_REF)
        samples.append(item);labels.append((int(row.No),cond))
result=evaluate(dataset=EvaluationDataset.from_list(samples),metrics=metrics,llm=llm,embeddings=embeddings,raise_exceptions=False)
scores=result.to_pandas()
scores.insert(0,'No',[a for a,b in labels]);scores.insert(1,'condition',[b for a,b in labels])
scores.to_csv('/content/prism_ragas_5metrics_results.csv',index=False,encoding='utf-8-sig')
metric_names=[m.name for m in metrics]
summary=scores.groupby('condition')[metric_names].agg(['count','mean','std'])
summary.to_csv('/content/prism_ragas_5metrics_summary.csv',encoding='utf-8-sig')
display(summary)
print('Evaluation rows containing NaN:',scores[metric_names].isna().any(axis=1).sum())

## 5. Download the CSV results


In [ ]:
from google.colab import files
for f in ['prism_evidence_audit.csv','prism_ragas_5metrics_results.csv','prism_ragas_5metrics_summary.csv']:
    files.download('/content/'+f)

### Interpretation notes

1. Faithfulness measures the proportion of answer claims supported by the selected evidence; it is **not** the accuracy across 30 questions.
2. Answer Correctness is an automated metric; determining strict accuracy requires human verification.
3. Incorrectly selected evidence can distort Faithfulness and the context metrics. Review `prism_evidence_audit.csv` first.
4. Identical Context Precision/Recall scores for RAG and non-RAG can be expected because this evaluation uses the same questions, references, and candidate evidence for both conditions. Do **not** interpret these values as PRISM retriever performance.
5. Compare RAG and non-RAG responses using matched questions, and report the number of non-missing observations after excluding NaN values.
